# 🧠 MESA — Memory, Epistemic & Salience Architecture
## Zero-Cost Colab Demo (No API Keys Required)

This notebook demonstrates MESA's core capabilities using **exclusively local resources**:
- **LLM**: Ollama + `llama3.2:3b` (runs on Colab's free CPU/GPU)
- **Embeddings**: `sentence-transformers/all-MiniLM-L6-v2` (local, 22 MB)
- **Graph DB**: KuzuDB (embedded, zero-config)
- **Vector DB**: LanceDB (embedded, zero-config)

**Total API cost: $0.00**

---
## Cell 1 — Install MESA

In [ ]:
!apt-get update
!apt-get install -y zstd
# Clone MESA repository (Replace Yasou13 with your actual handle)
!git clone https://github.com/Yasou13/MESA.git
%cd MESA
!pip install -r requirements-core.txt

---
## Cell 2 — Install & Start Ollama

In [ ]:
import os
import time

# Explicitly set environment variables BEFORE any MESA module is imported
os.environ["MESA_ZERO_COST_MODE"] = "true"
os.environ["MESA_OLLAMA_URL"] = "http://localhost:11434"
os.environ["MESA_LLM_PROVIDER"] = "ollama"
os.environ["LLM_MODEL_NAME"] = "llama3.2:3b"
os.environ["MESA_REBEL_ENABLED"] = "false"  # Skip REBEL on Colab free tier
os.environ["MESA_EXTRACTION_LANG"] = "en"

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background using nohup and redirect all outputs to /dev/null
!nohup ollama serve > /dev/null 2>&1 &
time.sleep(5)  # Strict sleep timer to ensure REST API is responsive
print('✅ Ollama server started')

# Pull the lightweight model
!ollama pull llama3.2:3b
print('✅ llama3.2:3b model ready')

---
## Cell 2.1 — Verify Ollama Installation

Quick sanity checks: version, API health, model list, and a test prompt.

In [ ]:
# Verify Ollama version
!ollama --version

# Check API health
!curl http://127.0.0.1:11434/api/tags

# List installed models
!ollama list

# Quick smoke test
!ollama run llama3.2:3b "who are you"

---
## Cell 3 — Full MESA Demo: Ingest, Query & Visualise Graph

This cell demonstrates MESA's end-to-end pipeline:
1. **Ingest** 10 records with temporal contradictions
2. **Query** to verify contradiction resolution
3. **Visualise** the knowledge graph structure

In [ ]:
import os
import shutil

from mesa_storage.dao import MemoryDAO
from mesa_storage.schemas import initialize_schema
from mesa_storage.sqlite_engine import AsyncEngine
from mesa_storage.vector_engine import VectorEngine

# ── Storage setup ──────────────────────────────────────────────────────────
# Use Colab local disk (LanceDB needs POSIX rename — Google Drive FUSE doesn't support it)
STORAGE_DIR = '/content/mesa_storage'
if os.path.exists(STORAGE_DIR):
    shutil.rmtree(STORAGE_DIR)
os.makedirs(STORAGE_DIR, exist_ok=True)
os.environ['MESA_STORAGE_PATH'] = STORAGE_DIR

# ── Import MESA components ─────────────────────────────────────────────────

# ── Demo dataset: 10 records with temporal contradictions ──────────────────
DEMO_RECORDS = [
    # Pair 1: Company HQ location (contradiction)
    {"content": "Acme Corp headquarters is located in San Francisco.",
     "agent_id": "demo", "metadata": {"source": "2023_annual_report", "timestamp": "2023-01-15"}},
    {"content": "Acme Corp relocated its headquarters to Austin, Texas in 2024.",
     "agent_id": "demo", "metadata": {"source": "2024_press_release", "timestamp": "2024-03-01"}},

    # Pair 2: CEO identity (contradiction)
    {"content": "Jane Smith is the CEO of Acme Corp.",
     "agent_id": "demo", "metadata": {"source": "linkedin", "timestamp": "2023-06-01"}},
    {"content": "John Doe was appointed CEO of Acme Corp, replacing Jane Smith.",
     "agent_id": "demo", "metadata": {"source": "board_minutes", "timestamp": "2024-01-15"}},

    # Pair 3: Product pricing (contradiction)
    {"content": "Acme Widget Pro is priced at $99.99.",
     "agent_id": "demo", "metadata": {"source": "product_catalog_v1", "timestamp": "2023-09-01"}},
    {"content": "Acme Widget Pro price increased to $149.99 effective January 2024.",
     "agent_id": "demo", "metadata": {"source": "pricing_update", "timestamp": "2024-01-01"}},

    # Non-contradictory supporting facts
    {"content": "Acme Corp was founded in 2010 in Silicon Valley.",
     "agent_id": "demo", "metadata": {"source": "wiki", "timestamp": "2023-01-01"}},
    {"content": "Acme Corp has 500 employees across 3 offices.",
     "agent_id": "demo", "metadata": {"source": "hr_system", "timestamp": "2024-02-01"}},
    {"content": "Acme Corp's primary product line includes Widget Pro and Widget Lite.",
     "agent_id": "demo", "metadata": {"source": "product_catalog_v2", "timestamp": "2024-01-15"}},
    {"content": "Acme Corp reported $50M revenue in fiscal year 2023.",
     "agent_id": "demo", "metadata": {"source": "sec_filing", "timestamp": "2024-03-15"}},
]


async def run_demo():
    # ── 1. Initialize storage engines ──────────────────────────────────
    print('\n🔧 Initializing MESA storage engines...')
    sqlite = AsyncEngine(db_path=f'{STORAGE_DIR}/mesa.db')
    await sqlite.initialize()
    await initialize_schema(sqlite)
    print('   ✅ SQLite WAL engine ready')

    vector = VectorEngine(uri=f'{STORAGE_DIR}/vector.lance')
    await vector.initialize()
    print('   ✅ LanceDB vector engine ready')

    dao = MemoryDAO(sqlite_engine=sqlite, vector_engine=vector)
    await dao.initialize()
    print('   ✅ MemoryDAO wired')

    # ── 2. Ingest records ─────────────────────────────────────────────
    print(f'\n📥 Ingesting {len(DEMO_RECORDS)} records...')
    for i, record in enumerate(DEMO_RECORDS, 1):
        embedding = await vector.compute_embedding(record['content'])
        entity_name = record.get('metadata', {}).get('source', f'Record {i}')
        _ = await dao.insert_memory(
            content=record['content'],
            agent_id=record['agent_id'],
            entity_name=entity_name,
            embedding=embedding,
        )
        status = '🔴 contradiction' if i <= 6 and i % 2 == 0 else '🟢 fact'
        print(f'   [{i:2d}/10] {status} → {record["content"][:60]}...')

    # ── 3. Query with contradiction detection ─────────────────────────
    QUERIES = [
        'Where is Acme Corp headquarters located?',
        'Who is the CEO of Acme Corp?',
        'What is the price of Acme Widget Pro?',
    ]

    print('\n🔍 Querying MESA (contradiction resolution)...')
    for query in QUERIES:
        query_vec = await vector.compute_embedding(query)
        results = await dao.search_memory(
            query_vector=query_vec,
            agent_id='demo',
            limit=3,
            include_graph=True,
        )
        print(f'\n   Q: {query}')
        if results:
            for j, r in enumerate(results, 1):
                content = r.get('graph', {}).get('entity_name', '') or r.get('node_id', '')[:80]
                print(f'   A{j}: {content}')
        else:
            print('   (no results)')

    # ── 4. Show graph structure ───────────────────────────────────────
    print('\n📊 Knowledge Graph Summary:')
    try:
        row = await sqlite.fetch_one('SELECT COUNT(*) as cnt FROM memory_nodes')
        count = row['cnt'] if row else 0
        print(f'   Nodes: {count}')
    except Exception as e:
        print(f'   (Could not query graph: {e})')

    # Cleanup
    await sqlite.close()
    print('\n✅ Demo complete!')


# Run
await run_demo()

---
## What just happened?

1. **Ingestion**: MESA stored 10 memory nodes, each with vector embeddings (local `all-MiniLM-L6-v2`) and SQLite metadata.
2. **Contradiction pairs**: Records 1-6 contain 3 temporal contradictions (HQ location, CEO, pricing). MESA's graph topology tracks temporal precedence.
3. **Query**: MESA's hybrid retrieval (vector similarity + graph routing) surfaces the *most recent* fact for each contradiction pair.

### Next Steps
- Try `MESA_ZERO_COST_MODE=true` with a larger dataset
- Run the full benchmark suite: `python scripts/run_ablation.py`
- See the [MESA Whitepaper](../benchmarks/FINAL_MESA_WHITEPAPER.md) for architecture details